# Step 5: QC Metrics
Compute and visualise quality metrics for all matched pairs.
Output: `{subject}_coreg_table_with_metrics.csv` → `/results/`

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
sys.path.insert(0, str(Path('.')))

from coreg_data_loading import (
    find_coreg_dirs, parse_coreg_dir_name, load_subject_data,
    load_landmarks
)
from coreg_matching import fit_tps, project_czstack_to_hcr
from coreg_metrics import (
    compute_tps_loo_residuals, compute_match_metrics
)
from coreg_alignment import CZ_RESOLUTION_UM

%load_ext autoreload
%autoreload 2

DATA_DIR = Path('/root/capsule/data')
SCRATCH_DIR = Path('/root/capsule/scratch')
RESULTS_DIR = Path('/root/capsule/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## Parameters

In [ ]:
subject_id = '790322'
czstack_date = '2025-07-10'
gfp_threshold_counts = 5


## Load Data

In [ ]:
coreg_dirs = find_coreg_dirs(DATA_DIR)
coreg_dir = None
for d in coreg_dirs:
    sid, cdate = parse_coreg_dir_name(d)
    if sid == subject_id and cdate == czstack_date:
        coreg_dir = d
        break

save_dir = SCRATCH_DIR / f'{subject_id}_{czstack_date}_coreg_cpsam'
if coreg_dir is None:
    coreg_dir = save_dir

data = load_subject_data(coreg_dir, subject_id, czstack_date,
                          gfp_threshold=gfp_threshold_counts, load_iter_paths=True)
czstack_df = data['czstack_df']
hcr_df = data['hcr_df']
hcr_scales = data['hcr_scales']
hcr_metrics = data['hcr_metrics']
spot_counts = data['spot_counts']


In [ ]:
# Load final landmarks and match metadata
final_lm_path = save_dir / f'{subject_id}_{czstack_date}_landmarks_auto_final.csv'
match_meta_path = save_dir / f'{subject_id}_{czstack_date}_auto_match_metadata.csv'

if final_lm_path.exists():
    final_landmarks = load_landmarks(final_lm_path)
else:
    # Fall back to manual landmarks
    lm_files = sorted(coreg_dir.glob('*landmarks*qced*.csv'))
    if not lm_files:
        lm_files = sorted(coreg_dir.glob('*landmarks*.csv'))
    final_landmarks = load_landmarks(lm_files[-1])
    print(f'Using manual landmarks: {lm_files[-1].name}')

print(f'Landmarks: {len(final_landmarks)} rows, {final_landmarks["active"].sum()} active')

if match_meta_path.exists():
    match_meta = pd.read_csv(match_meta_path)
    print(f'Match metadata: {len(match_meta)} auto matches')
else:
    match_meta = pd.DataFrame(columns=['czstack_cell_id', 'hcr_cell_id', 'iter_matched', 'match_probability'])
    print('No match metadata found (manual pipeline?)')


## Build Full Match Table

In [ ]:
# Extract all active matched pairs from final landmarks
def lm_to_match_table(lm_df):
    rows = []
    for _, r in lm_df[lm_df['active'] == True].iterrows():
        s = str(r['ids'])
        if 'cz' in s and '-hcr' in s:
            try:
                cz_id = int(s.split('-')[0].replace('cz', '').replace('qced_', ''))
                hcr_id = int(s.split('-')[1].replace('hcr', ''))
                rows.append({'czstack_cell_id': cz_id, 'hcr_cell_id': hcr_id})
            except Exception:
                pass
    return pd.DataFrame(rows)

all_pairs = lm_to_match_table(final_landmarks)
print(f'Total matched pairs: {len(all_pairs)}')

# Merge with metadata
if len(match_meta) > 0:
    all_pairs = all_pairs.merge(
        match_meta[['czstack_cell_id', 'hcr_cell_id', 'iter_matched', 'match_probability']],
        on=['czstack_cell_id', 'hcr_cell_id'],
        how='left'
    )
else:
    all_pairs['iter_matched'] = 0
    all_pairs['match_probability'] = np.nan


## Compute QC Metrics

In [ ]:
metrics_df = compute_match_metrics(
    matched_df=all_pairs,
    czstack_df=czstack_df,
    hcr_full_df=hcr_df,
    spot_counts=spot_counts,
    hcr_metrics=hcr_metrics,
    active_landmarks=final_landmarks,
    hcr_scales=hcr_scales,
    czstack_res_um=CZ_RESOLUTION_UM,
)

print('Metrics computed:')
print(metrics_df.describe())


In [ ]:
# TPS leave-one-out residuals (expensive but informative)
active_lm = final_landmarks[final_landmarks['active'] == True].copy().reset_index(drop=True)
loo_residuals = compute_tps_loo_residuals(active_lm, hcr_scales, CZ_RESOLUTION_UM)
print(f'LOO residuals (µm): median={loo_residuals.median():.2f}, mean={loo_residuals.mean():.2f}, '
      f'95th pct={loo_residuals.quantile(0.95):.2f}')


## QC Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. distance_um histogram per iteration
ax = axes[0, 0]
if 'iter_matched' in metrics_df.columns:
    for it in sorted(metrics_df['iter_matched'].dropna().unique()):
        sub = metrics_df[metrics_df['iter_matched'] == it]['distance_um'].dropna()
        ax.hist(sub, bins=40, alpha=0.5, label=f'Iter {int(it)}')
    ax.legend()
else:
    ax.hist(metrics_df['distance_um'].dropna(), bins=40, edgecolor='k')
ax.set_xlabel('TPS projection error (µm)')
ax.set_ylabel('Count')
ax.set_title('Projection error by iteration')

# 2. distance_um vs match_probability
ax = axes[0, 1]
if not metrics_df['match_probability'].isna().all():
    sc = ax.scatter(
        metrics_df['distance_um'], metrics_df['match_probability'],
        c=metrics_df.get('nn_rank', pd.Series(np.ones(len(metrics_df)))),
        cmap='viridis', s=10, alpha=0.6
    )
    plt.colorbar(sc, ax=ax, label='NN rank')
    ax.set_xlabel('distance_um (µm)')
    ax.set_ylabel('match_probability')
    ax.set_title('Distance vs match probability')
else:
    ax.hist(metrics_df['distance_um'].dropna(), bins=40, edgecolor='k')
    ax.set_title('Distance histogram')

# 3. LOO residual histogram
ax = axes[0, 2]
ax.hist(loo_residuals.dropna(), bins=40, edgecolor='k')
ax.set_xlabel('LOO TPS residual (µm)')
ax.set_ylabel('Count')
ax.set_title('TPS LOO residuals')

# 4. Cumulative match count per iteration
ax = axes[1, 0]
if 'iter_matched' in metrics_df.columns:
    iters = metrics_df.groupby('iter_matched').size().cumsum()
    ax.bar(iters.index, iters.values, edgecolor='k')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Cumulative matched count')
    ax.set_title('Matches per iteration')

# 5. GFP counts vs distance
ax = axes[1, 1]
if 'gfp_counts' in metrics_df.columns:
    ax.scatter(metrics_df['gfp_counts'], metrics_df['distance_um'],
               s=8, alpha=0.5)
    ax.set_xlabel('GFP spot counts')
    ax.set_ylabel('distance_um (µm)')
    ax.set_title('GFP counts vs projection error')

# 6. Match probability histogram
ax = axes[1, 2]
if not metrics_df['match_probability'].isna().all():
    ax.hist(metrics_df['match_probability'].dropna(), bins=40, edgecolor='k')
    ax.set_xlabel('match_probability')
    ax.set_ylabel('Count')
    ax.set_title('Match probability distribution')
else:
    ax.text(0.5, 0.5, 'No classifier\n(MNN-only mode)', ha='center', va='center',
            transform=ax.transAxes, fontsize=14)
    ax.set_title('Match probability (N/A)')

plt.suptitle(f'QC Metrics: {subject_id}', fontsize=14)
plt.tight_layout()
qc_plot_path = RESULTS_DIR / f'{subject_id}_{czstack_date}_qc_metrics.png'
plt.savefig(qc_plot_path, dpi=120)
plt.show()
print(f'QC plot saved to {qc_plot_path}')


In [ ]:
# 3D scatter: matched vs unmatched czstack cells
matched_cz_ids = set(all_pairs['czstack_cell_id'].values)
unmatched_cz = czstack_df[~czstack_df['czstack_cell_id'].isin(matched_cz_ids)]
matched_cz = czstack_df[czstack_df['czstack_cell_id'].isin(matched_cz_ids)]

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(unmatched_cz['czstack_x'], unmatched_cz['czstack_y'], unmatched_cz['czstack_z'],
           s=4, alpha=0.3, c='gray', label='Unmatched')

if not metrics_df['match_probability'].isna().all():
    probs = metrics_df.set_index('czstack_cell_id').reindex(matched_cz['czstack_cell_id'].values)['match_probability'].values
    sc = ax.scatter(matched_cz['czstack_x'], matched_cz['czstack_y'], matched_cz['czstack_z'],
                    s=8, alpha=0.8, c=probs, cmap='RdYlGn', label='Matched')
    plt.colorbar(sc, ax=ax, label='match_probability')
else:
    ax.scatter(matched_cz['czstack_x'], matched_cz['czstack_y'], matched_cz['czstack_z'],
               s=8, alpha=0.8, c='green', label='Matched')

ax.set_xlabel('X (px)')
ax.set_ylabel('Y (px)')
ax.set_zlabel('Z (px)')
ax.legend()
ax.set_title(f'3D Match Coverage: {subject_id}\n{len(matched_cz)}/{len(czstack_df)} matched')
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'{subject_id}_{czstack_date}_3d_coverage.png', dpi=100)
plt.show()


## Save Final Table

In [ ]:
# Merge metrics back into coreg table
final_table = all_pairs.merge(metrics_df, on=['czstack_cell_id', 'hcr_cell_id'], how='left',
                               suffixes=('', '_metrics'))

# Keep essential columns
keep_cols = ['czstack_cell_id', 'hcr_cell_id', 'iter_matched', 'match_probability',
             'distance_um', 'gfp_counts', 'gfp_density', 'hcr_volume_vox']
keep_cols = [c for c in keep_cols if c in final_table.columns]
final_table = final_table[keep_cols]

out_path = RESULTS_DIR / f'{subject_id}_coreg_table_with_metrics.csv'
final_table.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {final_table.shape}')
print(final_table.describe())


## Summary

In [ ]:
n_cz = len(czstack_df)
n_matched = len(all_pairs)
print(f'\n=== QC Summary: {subject_id} ===')
print(f'Total czstack cells:  {n_cz}')
print(f'Matched pairs:        {n_matched} ({100*n_matched/n_cz:.1f}%)')
print(f'Unmatched czstack:    {n_cz - n_matched}')
if 'distance_um' in metrics_df.columns:
    d = metrics_df['distance_um'].dropna()
    print(f'Projection error (µm): median={d.median():.2f}, mean={d.mean():.2f}')
if not loo_residuals.isna().all():
    print(f'LOO residual (µm):    median={loo_residuals.median():.2f}, mean={loo_residuals.mean():.2f}')
